In [1]:
!pip install numpy pandas scikit-learn xgboost joblib

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB 217.9 kB/s eta 0:07:47
   ---------------------------------------- 0.1/101.7 MB 328.2 kB/s eta 0:05:10
   ---------------------------------------- 0.2/101.7 MB 654.6 kB/s eta 0:02:36
   ---------------------------------------- 0.2/101.7 MB 846.9 kB/s eta 0:02:00
   ---------------------------------------- 0.3/101.7 MB 1.0 MB/s eta 0:01:37
   ---------------------------------------- 0.4/101.7 MB 1.1 MB/s eta 0:01:29
   ---------------------------------------- 0.6/101.7 MB 1.5 MB/s eta 0:01:08
   ---------------------------------------- 0.9/101.7 MB 2.0 MB/s eta 0:00:51
   ---------------------------------------- 1.1/101.7 MB 2.3 MB/s eta 0:00:45
   ---------------------------------------- 1.1/101.7 MB 2.3 MB/s eta 0:00:45

In [2]:
# Nếu thiếu package thì bỏ comment:
# %pip install -q xgboost joblib

import json, random, warnings, joblib
import numpy as np
import pandas as pd
import xgboost as xgb

from copy import deepcopy
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.preprocessing import LabelEncoder, Normalizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

DATA_PATH = r"D:\Tài Liệu Môn Học\Thay thế\annotated_clauses_topics.csv"
MODEL_OUTPUT_PATH = "xgboost_clause_code_model.pkl"
N_ITER_SEARCH = 20
EARLY_STOPPING_ROUNDS = 20

# 1) Load data
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", list(df.columns))

if "Clause" not in df.columns or "code" not in df.columns:
    raise ValueError("CSV phải có cột 'Clause' và 'code'.")

data = df[["Clause", "code"]].dropna().copy()
data["Clause"] = data["Clause"].astype(str).str.strip()
data["code"] = data["code"].astype(str).str.strip()
data = data[(data["Clause"] != "") & (data["code"] != "")].reset_index(drop=True)

display(data.head())
display(data["code"].value_counts().to_frame("count"))

# 2) Split 80 / 10 / 10
X = data["Clause"]
y = data["code"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print("Train:", len(X_train), "Valid:", len(X_valid), "Test:", len(X_test))

# 3) Encode labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_valid_enc = le.transform(y_valid)
y_test_enc = le.transform(y_test)
class_names = list(le.classes_)
num_classes = len(class_names)
print("Classes:", class_names)

# 4) Helpers
def build_features(vectorizer, svd, normalizer, train_texts, other_texts=None):
    X_train_tfidf = vectorizer.fit_transform(train_texts)
    actual_components = min(svd.n_components, max(2, X_train_tfidf.shape[1] - 1))
    if actual_components != svd.n_components:
        svd = TruncatedSVD(n_components=actual_components, random_state=RANDOM_STATE)

    X_train_red = normalizer.fit_transform(svd.fit_transform(X_train_tfidf))
    X_other_red = None
    if other_texts is not None:
        X_other_red = normalizer.transform(svd.transform(vectorizer.transform(other_texts)))
    return X_train_red, X_other_red, vectorizer, svd, normalizer

def transform_texts(vectorizer, svd, normalizer, texts):
    return normalizer.transform(svd.transform(vectorizer.transform(texts)))

def overall_metrics(y_true, y_pred):
    p_ma, r_ma, f_ma, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p_we, r_we, f_we, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(p_ma),
        "recall_macro": float(r_ma),
        "f1_macro": float(f_ma),
        "precision_weighted": float(p_we),
        "recall_weighted": float(r_we),
        "f1_weighted": float(f_we),
    }

def per_class_metrics(y_true, y_pred):
    rep = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
    rows = []
    for c in class_names:
        rows.append({
            "class": c,
            "precision": float(rep[c]["precision"]),
            "recall": float(rep[c]["recall"]),
            "f1_score": float(rep[c]["f1-score"]),
            "support": int(rep[c]["support"]),
        })
    return pd.DataFrame(rows), rep

def eval_split(name, model, vectorizer, svd, normalizer, texts, y_true):
    Xf = transform_texts(vectorizer, svd, normalizer, texts)
    y_pred = model.predict(Xf).astype(int)
    per_cls, rep = per_class_metrics(y_true, y_pred)
    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred),
        index=[f"true_{c}" for c in class_names],
        columns=[f"pred_{c}" for c in class_names]
    )
    return {"overall": {"split": name, **overall_metrics(y_true, y_pred)}, "per_class": per_cls, "report": rep, "cm": cm}

# 5) Hyperparameter tuning
param_distributions = {
    # TF-IDF
    "max_features": [12000, 18000],
    "ngram_range": [(1, 1), (1, 2)],
    "min_df": [3, 5],
    "max_df": [0.95],

    # SVD
    "svd_components": [150, 200],

    # XGBoost
    "n_estimators": [300, 500],
    "learning_rate": [0.05, 0.08],
    "max_depth": [3, 4],
    "min_child_weight": [5, 7],
    "subsample": [0.80],
    "colsample_bytree": [0.70],
    "gamma": [0.3, 0.7],
    "reg_alpha": [0.5, 1.0],
    "reg_lambda": [5.0, 10.0],
}
sampled_params = list(
    ParameterSampler(
        param_distributions,
        n_iter=N_ITER_SEARCH,
        random_state=RANDOM_STATE
    )
)

def run_trial(params):
    vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    sublinear_tf=True,
    stop_words="english",
    max_features=params["max_features"],
    ngram_range=params["ngram_range"],
    min_df=params["min_df"],
    max_df=params["max_df"]
    )
    svd = TruncatedSVD(n_components=params["svd_components"], random_state=RANDOM_STATE)
    normalizer = Normalizer(copy=False)

    Xtr, Xva, vectorizer, svd, normalizer = build_features(vectorizer, svd, normalizer, X_train, X_valid)
    model = xgb.XGBClassifier(
        objective="multi:softprob", num_class=num_classes,
        n_estimators=params["n_estimators"], learning_rate=params["learning_rate"],
        max_depth=params["max_depth"], min_child_weight=params["min_child_weight"],
        subsample=params["subsample"], colsample_bytree=params["colsample_bytree"],
        gamma=params["gamma"], reg_alpha=params["reg_alpha"], reg_lambda=params["reg_lambda"],
        tree_method="hist", eval_metric="mlogloss", early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    sw = compute_sample_weight(class_weight="balanced", y=y_train_enc)
    model.fit(Xtr, y_train_enc, sample_weight=sw, eval_set=[(Xva, y_valid_enc)], verbose=False)

    tr_pred = model.predict(Xtr).astype(int)
    va_pred = model.predict(Xva).astype(int)
    tr = overall_metrics(y_train_enc, tr_pred)
    va = overall_metrics(y_valid_enc, va_pred)
    gap_f1 = tr["f1_macro"] - va["f1_macro"]
    gap_acc = tr["accuracy"] - va["accuracy"]
    score = (
         1.00 * va["f1_macro"]
         + 0.20 * va["f1_weighted"]
         - 0.50 * max(gap_f1, 0)
         - 0.15 * max(gap_acc, 0)
         )
    if gap_f1 > 0.03:
        score -= 0.03
    if gap_f1 > 0.05:
        score -= 0.05

    return {
        "params": deepcopy(params),
        "vectorizer": vectorizer, "svd": svd, "normalizer": normalizer, "model": model,
        "selection_score": float(score),
        "train_accuracy": tr["accuracy"], "train_f1_macro": tr["f1_macro"], "train_f1_weighted": tr["f1_weighted"],
        "valid_accuracy": va["accuracy"], "valid_f1_macro": va["f1_macro"], "valid_f1_weighted": va["f1_weighted"],
        "overfit_gap_macro_f1": float(gap_f1),
        "best_iteration": int(getattr(model, "best_iteration", params["n_estimators"] - 1)) + 1,
        "actual_svd_components": int(svd.n_components),
    }

trial_results = []
for i, params in enumerate(sampled_params, start=1):
    print(f"Trial {i}/{len(sampled_params)}")
    trial_results.append(run_trial(params))

tuning_df = pd.DataFrame([
    {
        **r["params"],
        "selection_score": r["selection_score"],
        "train_accuracy": r["train_accuracy"],
        "train_f1_macro": r["train_f1_macro"],
        "train_f1_weighted": r["train_f1_weighted"],
        "valid_accuracy": r["valid_accuracy"],
        "valid_f1_macro": r["valid_f1_macro"],
        "valid_f1_weighted": r["valid_f1_weighted"],
        "overfit_gap_macro_f1": r["overfit_gap_macro_f1"],
        "best_iteration": r["best_iteration"],
        "actual_svd_components": r["actual_svd_components"],
    } for r in trial_results
]).sort_values(by=["selection_score", "valid_f1_macro", "valid_f1_weighted"], ascending=False).reset_index(drop=True)

display(tuning_df.head(10))

best = sorted(trial_results, key=lambda r: (r["selection_score"], r["valid_f1_macro"], r["valid_f1_weighted"]), reverse=True)[0]
best_params = best["params"]
best_vectorizer = best["vectorizer"]
best_svd = best["svd"]
best_normalizer = best["normalizer"]
best_model = best["model"]

print("BEST PARAMS:")
print(json.dumps(best_params, indent=2, default=str))
print("Best boosting rounds:", best["best_iteration"])
print("Best SVD components :", best["actual_svd_components"])

# 6) Final evaluation
train_eval = eval_split("train", best_model, best_vectorizer, best_svd, best_normalizer, X_train, y_train_enc)
valid_eval = eval_split("valid", best_model, best_vectorizer, best_svd, best_normalizer, X_valid, y_valid_enc)
test_eval  = eval_split("test",  best_model, best_vectorizer, best_svd, best_normalizer, X_test,  y_test_enc)

overall_df = pd.DataFrame([train_eval["overall"], valid_eval["overall"], test_eval["overall"]])
display(overall_df)

print("TRAIN - per class")
display(train_eval["per_class"])
print("VALID - per class")
display(valid_eval["per_class"])
print("TEST - per class")
display(test_eval["per_class"])

print("TRAIN confusion matrix")
display(train_eval["cm"])
print("VALID confusion matrix")
display(valid_eval["cm"])
print("TEST confusion matrix")
display(test_eval["cm"])

# 7) Save model bundle to .pkl
bundle = {
    "model_name": "TFIDF + TruncatedSVD + XGBoost",
    "text_column": "Clause",
    "target_column": "code",
    "random_state": RANDOM_STATE,
    "class_names": class_names,
    "label_encoder": le,
    "vectorizer": best_vectorizer,
    "svd": best_svd,
    "normalizer": best_normalizer,
    "model": best_model,
    "best_params": best_params,
    "best_iteration": best["best_iteration"],
    "actual_svd_components": best["actual_svd_components"],
    "split_info": {"train_size": int(len(X_train)), "valid_size": int(len(X_valid)), "test_size": int(len(X_test))},
    "metrics": {
        "train_overall": train_eval["overall"],
        "valid_overall": valid_eval["overall"],
        "test_overall": test_eval["overall"],
        "train_per_class": train_eval["per_class"].to_dict(orient="records"),
        "valid_per_class": valid_eval["per_class"].to_dict(orient="records"),
        "test_per_class": test_eval["per_class"].to_dict(orient="records"),
    }
}
joblib.dump(bundle, MODEL_OUTPUT_PATH)
print(f"Saved model bundle to: {MODEL_OUTPUT_PATH}")

# 8) Predict new clauses
def predict_new_clauses(clauses, model_path=MODEL_OUTPUT_PATH):
    bundle = joblib.load(model_path)
    texts = pd.Series(clauses, dtype="object").fillna("").astype(str)
    Xf = transform_texts(bundle["vectorizer"], bundle["svd"], bundle["normalizer"], texts)
    pred_enc = bundle["model"].predict(Xf).astype(int)
    pred_lbl = bundle["label_encoder"].inverse_transform(pred_enc)
    proba = bundle["model"].predict_proba(Xf).max(axis=1)
    return pd.DataFrame({"Clause": texts, "Predicted_code": pred_lbl, "Confidence": proba})

demo = ["Only stayed for 15mins", "Take a scooter from Hoi An", "Great places to chill all day"]
predict_new_clauses(demo)

Shape: (23173, 10)
Columns: ['review_id', 'Username', 'Address', 'Rating', 'Language', 'ID_clause', 'Clause', 'topic', 'code', 'topic_name']


,Clause,code
0,The lake is as green as paint.,EL
1,but the 12km hike (there and back) was well wo...,EL
2,It was quite crowded,TCr
3,but they actually physically blocked us from L...,SE
4,Example rice sheets for roll in the market 800...,TC


,count
code,
VE,9017
TC,1872
LC,1734
RA,1697
HL,1477
CT,1475
SE,1411
UH,1102
TCr,984


Train: 18538 Valid: 2317 Test: 2318
Classes: ['CA', 'CT', 'EC', 'EL', 'HL', 'LC', 'RA', 'SE', 'TC', 'TCr', 'UH', 'VE', 'VR']
Trial 1/20
Trial 2/20
Trial 3/20
Trial 4/20
Trial 5/20
Trial 6/20
Trial 7/20
Trial 8/20
Trial 9/20
Trial 10/20
Trial 11/20
Trial 12/20
Trial 13/20
Trial 14/20
Trial 15/20
Trial 16/20
Trial 17/20
Trial 18/20
Trial 19/20
Trial 20/20


,svd_components,subsample,reg_lambda,reg_alpha,ngram_range,n_estimators,min_df,min_child_weight,max_features,max_df,...,selection_score,train_accuracy,train_f1_macro,train_f1_weighted,valid_accuracy,valid_f1_macro,valid_f1_weighted,overfit_gap_macro_f1,best_iteration,actual_svd_components
0,200,0.8,10.0,0.5,"(1, 1)",300,5,7,12000,0.95,...,0.582910,0.731147,0.719376,0.733716,0.646957,0.602753,0.655485,0.116623,300,200
1,200,0.8,10.0,1.0,"(1, 1)",300,3,7,12000,0.95,...,0.577512,0.726346,0.715378,0.728550,0.641778,0.598443,0.651110,0.116935,300,200
2,200,0.8,10.0,1.0,"(1, 1)",500,3,7,12000,0.95,...,0.575081,0.793343,0.790325,0.793470,0.669400,0.622287,0.677022,0.168038,500,200
3,200,0.8,10.0,1.0,"(1, 1)",500,3,5,18000,0.95,...,0.571194,0.863578,0.866763,0.862919,0.697454,0.645950,0.702844,0.220813,500,200
4,150,0.8,10.0,1.0,"(1, 1)",300,5,7,12000,0.95,...,0.568967,0.702233,0.689430,0.705547,0.623651,0.585731,0.634369,0.103699,300,150
5,200,0.8,10.0,1.0,"(1, 1)",300,5,7,12000,0.95,...,0.568633,0.885910,0.891718,0.885198,0.711696,0.651802,0.714604,0.239916,300,200
6,200,0.8,5.0,0.5,"(1, 2)",500,5,7,12000,0.95,...,0.567348,0.804348,0.800017,0.804823,0.675011,0.620180,0.682438,0.179837,500,200
7,200,0.8,10.0,0.5,"(1, 1)",500,5,5,18000,0.95,...,0.557306,0.896914,0.902140,0.896233,0.711696,0.648770,0.715021,0.253370,500,200
8,150,0.8,10.0,1.0,"(1, 1)",300,5,7,12000,0.95,...,0.556526,0.758928,0.753885,0.759539,0.644368,0.599889,0.654097,0.153996,300,150
9,150,0.8,10.0,0.5,"(1, 1)",300,3,5,18000,0.95,...,0.556287,0.696785,0.685893,0.699868,0.613725,0.577796,0.624990,0.108097,300,150


...

,pred_CA,pred_CT,pred_EC,pred_EL,pred_HL,pred_LC,pred_RA,pred_SE,pred_TC,pred_TCr,pred_UH,pred_VE,pred_VR
true_CA,71,0,0,2,4,0,1,1,0,0,2,7,2
true_CT,1,113,3,6,0,3,5,2,2,2,4,3,3
true_EC,0,2,23,4,0,2,2,2,3,2,1,5,0
true_EL,1,0,4,44,5,0,1,2,0,1,1,11,0
true_HL,5,0,0,0,118,1,2,5,0,1,9,6,1
true_LC,0,9,3,5,1,131,2,2,1,0,7,8,4
true_RA,6,1,7,9,1,1,112,1,1,9,1,19,1
true_SE,2,9,1,2,2,9,5,83,6,3,7,12,0
true_TC,4,6,2,2,2,11,9,3,126,7,5,10,0
true_TCr,0,3,0,4,2,4,1,6,0,66,2,9,2


TEST confusion matrix


,pred_CA,pred_CT,pred_EC,pred_EL,pred_HL,pred_LC,pred_RA,pred_SE,pred_TC,pred_TCr,pred_UH,pred_VE,pred_VR
true_CA,79,0,0,7,0,1,0,0,0,0,1,1,1
true_CT,2,104,5,10,1,3,5,1,0,5,2,8,2
true_EC,0,3,27,1,1,2,0,1,0,3,1,5,2
true_EL,1,2,0,42,1,0,1,0,0,4,1,15,3
true_HL,4,1,0,1,108,7,4,6,1,0,5,10,0
true_LC,3,1,6,3,2,128,4,5,7,4,3,6,2
true_RA,6,1,5,10,2,2,107,4,4,4,2,22,1
true_SE,4,9,2,2,2,8,10,72,5,2,12,13,0
true_TC,1,7,1,2,1,14,5,4,129,10,4,4,5
true_TCr,2,5,0,2,2,3,3,2,0,69,4,3,3


Saved model bundle to: xgboost_clause_code_model.pkl


,Clause,Predicted_code,Confidence
0,Only stayed for 15mins,SE,0.388333
1,Take a scooter from Hoi An,VE,0.241299
2,Great places to chill all day,VE,0.207900
